# conv-leakyrelu-block-discriminator — worked example 3: Stack two downsampling blocks to go 32x32 to 8x8

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-leakyrelu-block-discriminator`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

DCGAN discriminators stack several `Conv(stride=2)->BN->LeakyReLU` blocks; each block halves spatial size and doubles channels. Two stacked blocks take 32×32 down to 8×8. Reasoning about the composed shape is just applying the halving rule twice.

## Worked solution

**Step 1 — a reusable block factory.** Write a helper that returns the canonical `Conv(kernel=4, stride=2, padding=1, bias=False)->BatchNorm2d->LeakyReLU(0.2)` block. Each call halves H and W.

**Step 2 — first stacked block.** `64 -> 128` channels takes 32×32 to 16×16 (`floor((32+2-4)/2)+1 = 16`).

**Step 3 — second stacked block.** `128 -> 256` channels takes 16×16 to 8×8 (`floor((16+2-4)/2)+1 = 8`).

**Step 4 — compose with nn.Sequential.** Nesting the two blocks inside an outer `nn.Sequential` gives one module. Push `(2, 64, 32, 32)` through it and expect `(2, 256, 8, 8)`. We also confirm every conv has `bias=None` since each is followed by BatchNorm.

In [ ]:
import torch.nn as nn

def ds_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.LeakyReLU(0.2, inplace=True),
    )

def build_two_block_stack():
    return nn.Sequential(
        ds_block(64, 128),
        ds_block(128, 256),
    )

t.manual_seed(0)
net = build_two_block_stack()
x = t.randn(2, 64, 32, 32)
out = net(x)
print(tuple(out.shape))
convs = [m for m in net.modules() if isinstance(m, nn.Conv2d)]
print(all(c.bias is None for c in convs))